# 04 - Final comparison of all models

## How to reproduce

From the repository root (Python 3.11, uv installed):

```sh
uv sync --locked
uv run python -m weapon_detection.ml.compare
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/04_comparison.ipynb
```

compare.py only reads saved results (it re-checks every metric against the saved predictions). The fine-tuned YOLO row appears automatically after `yolo_eval.py` has been run on its checkpoint.
The measured runtime is printed in the last cell.

**Seed.**

In [1]:
# Fixed seed for everything random in this notebook (the modules use SEED = 42 too).
import random
import time

import numpy as np
import pandas as pd
from IPython.display import display

from weapon_detection.ml.common import OUTPUT_DIR, SEED

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.width", 200)
notebook_start = time.perf_counter()
print("SEED =", SEED)

SEED = 42


**Step 1 - Regenerate tables, figures and report numbers.** This also checks that every metric in the table can be recomputed from the saved prediction files.

In [2]:
from weapon_detection.ml import compare

compare.main()

Consistency check: 7 models x 2 splits recomputed, 0 mismatches
                                           model  macro_f1  neg_recall  accuracy  roc_auc
                               Majority baseline    0.4743      0.0000    0.9023   0.5000
               Logistic Regression (handcrafted)    0.5976      0.2778    0.8567   0.6371
                     Random Forest (handcrafted)    0.6367      0.3444    0.8719   0.7257
         Logistic Regression (handcrafted + HOG)    0.6299      0.4111    0.8480   0.7449
               Random Forest (handcrafted + HOG)    0.6729      0.3889    0.8893   0.7893
YOLOv8n pretrained (Open Images, not fine-tuned)    0.4262      0.9111    0.4777   0.6661
                    YOLOv8n fine-tuned (firearm)    0.8025      0.6778    0.9273   0.9211


                                        requirement  status
                                      >= 500 images     met
                           >= 6 meaningful features     met
                                   trivial baseline     met
    >= 3 models from >= 2 families (>= 1 classical)     met
          train/test split decided before modelling  partly
all preprocessing fitted on the training split only     met
         precision / recall / F1 + confusion matrix     met
                                      leakage audit     met
            EDA with a one-line takeaway per figure  partly
            error analysis (5-10 wrong predictions)  partly
    GUI that takes audience input and predicts live     met
                                 fixed random seeds     met
                               documented notebooks  partly
                          runs on CPU or free Colab     met
             AI tools listed honestly in the report not yet


Wrote tables, 14 figures, report_numbers.json and outputs/report/


**Step 2 - Image-level results on TEST and VALIDATION.** Firearm is the positive class; `neg_*` columns are the no-firearm class.

In [3]:
from IPython.display import Markdown

results = OUTPUT_DIR / 'results'
display(Markdown((results / 'final_test_table.md').read_text()))
display(Markdown((results / 'final_val_table.md').read_text()))

# Image-level results on the TEST split

Firearm = positive class. neg_* = no-firearm class. Threshold chosen on validation. ms/image: classical = model only on precomputed features; YOLO = read + inference.

| model | family | main_model | precision | recall | f1 | accuracy | balanced_accuracy | macro_f1 | neg_precision | neg_recall | neg_f1 | roc_auc | pr_auc | threshold | predict_ms_per_image |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Majority baseline | trivial (DummyClassifier) |  | 0.9023 | 1.0000 | 0.9486 | 0.9023 | 0.5000 | 0.4743 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.9023 |  | 0.0006 |
| Logistic Regression (handcrafted) | linear model |  | 0.9216 | 0.9194 | 0.9205 | 0.8567 | 0.5986 | 0.5976 | 0.2717 | 0.2778 | 0.2747 | 0.6371 | 0.9382 | 0.3200 | 0.0013 |
| Random Forest (handcrafted) | tree ensemble |  | 0.9290 | 0.9290 | 0.9290 | 0.8719 | 0.6367 | 0.6367 | 0.3444 | 0.3444 | 0.3444 | 0.7257 | 0.9494 | 0.4900 | 0.2478 |
| Logistic Regression (handcrafted + HOG) | linear model | yes | 0.9335 | 0.8953 | 0.9140 | 0.8480 | 0.6532 | 0.6299 | 0.2984 | 0.4111 | 0.3458 | 0.7449 | 0.9583 | 0.4700 | 0.0105 |
| Random Forest (handcrafted + HOG) | tree ensemble | yes | 0.9344 | 0.9434 | 0.9389 | 0.8893 | 0.6662 | 0.6729 | 0.4268 | 0.3889 | 0.4070 | 0.7893 | 0.9677 | 0.7600 | 0.2611 |
| YOLOv8n pretrained (Open Images, not fine-tuned) | CNN object detector |  | 0.9781 | 0.4308 | 0.5982 | 0.4777 | 0.6710 | 0.4262 | 0.1477 | 0.9111 | 0.2543 | 0.6661 | 0.9343 | 0.0100 | 107.7535 |
| YOLOv8n fine-tuned (firearm) | CNN object detector | yes | 0.9647 | 0.9543 | 0.9595 | 0.9273 | 0.8160 | 0.8025 | 0.6162 | 0.6778 | 0.6455 | 0.9211 | 0.9887 | 0.1200 | 146.6271 |


# Image-level results on the VAL split

Firearm = positive class. neg_* = no-firearm class. Threshold chosen on validation. ms/image: classical = model only on precomputed features; YOLO = read + inference.

| model | family | main_model | precision | recall | f1 | accuracy | balanced_accuracy | macro_f1 | neg_precision | neg_recall | neg_f1 | roc_auc | pr_auc | threshold | predict_ms_per_image |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Majority baseline | trivial (DummyClassifier) |  | 0.9023 | 1.0000 | 0.9486 | 0.9023 | 0.5000 | 0.4743 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.9023 |  | 0.0006 |
| Logistic Regression (handcrafted) | linear model |  | 0.9300 | 0.8797 | 0.9041 | 0.8317 | 0.6343 | 0.6076 | 0.2593 | 0.3889 | 0.3111 | 0.6621 | 0.9391 | 0.3200 | 0.0013 |
| Random Forest (handcrafted) | tree ensemble |  | 0.9362 | 0.9182 | 0.9271 | 0.8697 | 0.6702 | 0.6574 | 0.3585 | 0.4222 | 0.3878 | 0.7563 | 0.9611 | 0.4900 | 0.1478 |
| Logistic Regression (handcrafted + HOG) | linear model | yes | 0.9409 | 0.9194 | 0.9300 | 0.8751 | 0.6930 | 0.6761 | 0.3853 | 0.4667 | 0.4221 | 0.7732 | 0.9631 | 0.4700 | 0.0103 |
| Random Forest (handcrafted + HOG) | tree ensemble | yes | 0.9366 | 0.9422 | 0.9394 | 0.8903 | 0.6767 | 0.6811 | 0.4353 | 0.4111 | 0.4229 | 0.8021 | 0.9683 | 0.7600 | 0.1996 |
| YOLOv8n pretrained (Open Images, not fine-tuned) | CNN object detector |  | 0.9920 | 0.4465 | 0.6158 | 0.4973 | 0.7066 | 0.4445 | 0.1590 | 0.9667 | 0.2732 | 0.7039 | 0.9416 | 0.0100 | 100.6229 |
| YOLOv8n fine-tuned (firearm) | CNN object detector | yes | 0.9723 | 0.9711 | 0.9717 | 0.9490 | 0.8578 | 0.8560 | 0.7363 | 0.7444 | 0.7403 | 0.9173 | 0.9862 | 0.1200 | 149.7863 |


**Step 3 - Detection metrics are kept separate** (mAP is a different kind of number from image-level F1).

In [4]:
Markdown((OUTPUT_DIR / 'results' / 'detection_metrics.md').read_text())

# Detection metrics (separate from image-level metrics)

| checkpoint_key | checkpoint | split | imgsz | mAP50 | mAP50_95 | precision | recall |
| --- | --- | --- | --- | --- | --- | --- | --- |
| yolov8n_finetuned | models/firearm_yolov8n_best.pt | test | 640 | 0.7421 | 0.4298 | 0.7836 | 0.6602 |


**Step 4 - Comparison figures.** The figures are linked, not embedded, to keep this notebook small. Copies are in `report_assets/figures/` (`uv run python scripts/build_report_assets.py`); originals in `outputs/ml/figures/`.

In [5]:
from pathlib import Path

assets = Path('..') / 'report_assets' / 'figures'  # relative to notebooks/
names = [p.name for p in sorted((OUTPUT_DIR / 'figures').glob('fig_*.png'))]
display(Markdown('\n\n'.join(f'**{n}**\n\n![{n}]({(assets / n).as_posix()})' for n in names)))

**fig_01_class_balance.png**

![fig_01_class_balance.png](../report_assets/figures/fig_01_class_balance.png)

**fig_02_object_classes.png**

![fig_02_object_classes.png](../report_assets/figures/fig_02_object_classes.png)

**fig_03_firearm_box_area.png**

![fig_03_firearm_box_area.png](../report_assets/figures/fig_03_firearm_box_area.png)

**fig_04_image_sizes.png**

![fig_04_image_sizes.png](../report_assets/figures/fig_04_image_sizes.png)

**fig_05_feature_distributions.png**

![fig_05_feature_distributions.png](../report_assets/figures/fig_05_feature_distributions.png)

**fig_06_negative_sources_v1_v2.png**

![fig_06_negative_sources_v1_v2.png](../report_assets/figures/fig_06_negative_sources_v1_v2.png)

**fig_07_confusion_main_models_test.png**

![fig_07_confusion_main_models_test.png](../report_assets/figures/fig_07_confusion_main_models_test.png)

**fig_08_roc_test.png**

![fig_08_roc_test.png](../report_assets/figures/fig_08_roc_test.png)

**fig_09_pr_test.png**

![fig_09_pr_test.png](../report_assets/figures/fig_09_pr_test.png)

**fig_10_macro_f1_neg_recall_test.png**

![fig_10_macro_f1_neg_recall_test.png](../report_assets/figures/fig_10_macro_f1_neg_recall_test.png)

**fig_11_latency.png**

![fig_11_latency.png](../report_assets/figures/fig_11_latency.png)

**fig_12_errors_best_classical.png**

![fig_12_errors_best_classical.png](../report_assets/figures/fig_12_errors_best_classical.png)

**fig_13_errors_yolov8n_pretrained.png**

![fig_13_errors_yolov8n_pretrained.png](../report_assets/figures/fig_13_errors_yolov8n_pretrained.png)

**fig_14_errors_yolov8n_finetuned.png**

![fig_14_errors_yolov8n_finetuned.png](../report_assets/figures/fig_14_errors_yolov8n_finetuned.png)

**Step 5 - Course requirement checklist** (strict; generated from the files).

In [6]:
Markdown((OUTPUT_DIR / 'requirements_check.md').read_text())

# Course requirement checklist

Generated by `weapon_detection/ml/compare.py`.

| requirement | status | evidence | note |
| --- | --- | --- | --- |
| >= 500 images | met | outputs/ml/audit.json | 9425 images |
| >= 6 meaningful features | met | weapon_detection/ml/features.py | 10 handcrafted + 1764 HOG values |
| trivial baseline | met | outputs/ml/results/final_test_table.md | DummyClassifier(most_frequent) |
| >= 3 models from >= 2 families (>= 1 classical) | met | outputs/ml/results/final_test_table.md | trained on this data: Logistic Regression (linear model), Random Forest (tree ensemble), YOLOv8n fine-tuned (CNN object detector) |
| train/test split decided before modelling | partly | outputs/ml/audit.json (split_v2.note); weapon_detection/ml/split_v2.py | split fixed before modelling, but revised once (v2) after preliminary v1 test results; both versions documented |
| all preprocessing fitted on the training split only | met | weapon_detection/ml/classical.py; tests/test_ml_classical.py | StandardScaler inside Pipeline; per-image features need no fitting |
| precision / recall / F1 + confusion matrix | met | outputs/ml/results/final_test_table.md; outputs/ml/figures/fig_07_* |  |
| leakage audit | met | outputs/ml/leakage_audit.md; outputs/ml/audit.json |  |
| EDA with a one-line takeaway per figure | partly | outputs/ml/eda_notes.md; outputs/ml/figure_index.md | figures and factual notes exist; takeaways are for the student to write |
| error analysis (5-10 wrong predictions) | partly | outputs/ml/error_analysis*.csv; fig_12+ | wrong predictions listed; 'my_explanation' column still to be written |
| GUI that takes audience input and predicts live | met | predict_app.py; outputs/ml/gui_smoke_test.json |  |
| fixed random seeds | met | weapon_detection/ml/common.py (SEED = 42) |  |
| documented notebooks | partly | notebooks/ | 01_data_and_eda.ipynb: outputs stored, 02_classical_models.ipynb: outputs stored, 03_yolo_colab.ipynb: no stored outputs, 04_comparison.ipynb: no stored outputs |
| runs on CPU or free Colab | met | README.md (ML experiment); notebooks/03_yolo_colab.ipynb |  |
| AI tools listed honestly in the report | not yet | outputs/report/ai_use_disclosure_facts.md | student to confirm and write |


**Runtime.**

In [7]:
# Measured runtime of this notebook on the machine that executed it.
print(f"Notebook runtime: {time.perf_counter() - notebook_start:.0f} seconds")

Notebook runtime: 19 seconds
